<a href="https://colab.research.google.com/github/jahid-dev/-Thesis-Data-Driven-Student-Behavior-Modeling-and-Dropout-Prediction-Using-Advanced-ML-Techniques/blob/main/02_UPV_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SECTION 1: Environment Setup

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


# SECTION 2: Google Drive & Project Paths

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final"
)

DATA_DIR = BASE_DIR / "Datasets"

UPV_DIR = DATA_DIR / "UPV"

NOTEBOOK_DIR = BASE_DIR / "Notebooks"

RESULT_DIR = BASE_DIR / "Results"

AUDIT_RESULT_DIR = RESULT_DIR / "UPV" / "Audit"

PREPROCESSING_RESULT_DIR = RESULT_DIR / "UPV" / "Preprocessing"

PREPROCESSING_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("BASE_DIR:", BASE_DIR)
print("UPV_DIR:", UPV_DIR)
print("AUDIT_RESULT_DIR:", AUDIT_RESULT_DIR)
print("PREPROCESSING_RESULT_DIR:", PREPROCESSING_RESULT_DIR)

print("\nPath verification:")
print("BASE exists:", BASE_DIR.exists())
print("UPV exists:", UPV_DIR.exists())
print("Audit results exist:", AUDIT_RESULT_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final
UPV_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Datasets/UPV
AUDIT_RESULT_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Audit
PREPROCESSING_RESULT_DIR: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Preprocessing

Path verification:
BASE exists: True
UPV exists: True
Audit results exist: True


# SECTION 3: Load Raw UPV Datasets

In [ ]:
df_2018 = pd.read_csv(
    UPV_DIR / "dataset_2018_hash.csv",
    sep=";",
    decimal=","
)

df_2021 = pd.read_csv(
    UPV_DIR / "dataset_2021_hash.csv",
    sep=";",
    decimal=","
)

df_2022 = pd.read_csv(
    UPV_DIR / "dataset_2022_hash.csv",
    sep=";",
    decimal=","
)

print("Datasets loaded successfully.\n")

print("2018:", df_2018.shape)
print("2021:", df_2021.shape)
print("2022:", df_2022.shape)

Datasets loaded successfully.

2018: (152446, 178)
2021: (153120, 178)
2022: (159173, 169)


# SECTION 4: Load Audit Results

In [ ]:
# The audit outputs generated by 01_UPV_Data_Audit.ipynb
# are loaded here and used as the basis for preprocessing.
feature_summary = pd.read_csv(
    AUDIT_RESULT_DIR / "feature_summary_all.csv"
)

feature_group_summary = pd.read_csv(
    AUDIT_RESULT_DIR / "feature_group_summary.csv"
)

common_columns_df = pd.read_csv(
    AUDIT_RESULT_DIR / "common_columns.csv"
)

dataset_specific_columns_df = pd.read_csv(
    AUDIT_RESULT_DIR / "dataset_specific_columns.csv"
)

target_summary = pd.read_csv(
    AUDIT_RESULT_DIR / "target_summary.csv"
)

audit_decisions = pd.read_csv(
    AUDIT_RESULT_DIR / "audit_decisions.csv"
)

print("Audit results loaded successfully.\n")

print("Feature summary:", feature_summary.shape)
print("Feature group summary:", feature_group_summary.shape)
print("Common columns:", common_columns_df.shape)
print("Dataset-specific columns:", dataset_specific_columns_df.shape)
print("Target summary:", target_summary.shape)
print("Audit decisions:", audit_decisions.shape)

Audit results loaded successfully.

Feature summary: (525, 7)
Feature group summary: (36, 5)
Common columns: (70, 1)
Dataset-specific columns: (315, 2)
Target summary: (3, 7)
Audit decisions: (8, 4)


# SECTION 5: Verify Audit-to-Preprocessing Connection

In [ ]:
# Confirm that the audit results correspond to the raw datasets
# currently loaded in this notebook.
common_columns = common_columns_df["Column"].tolist()

print("=" * 70)
print("AUDIT CONNECTION CHECK")
print("=" * 70)

print("\nCommon columns according to audit:", len(common_columns))

for dataset_name, df in {
    "2018": df_2018,
    "2021": df_2021,
    "2022": df_2022
}.items():

    missing_from_dataset = [
        col for col in common_columns
        if col not in df.columns
    ]

    print(f"\nDataset {dataset_name}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing common columns:", len(missing_from_dataset))

    if missing_from_dataset:
        print(missing_from_dataset)

print("\nAudit connection check completed.")

AUDIT CONNECTION CHECK

Common columns according to audit: 70

Dataset 2018
Rows: 152446
Columns: 178
Missing common columns: 0

Dataset 2021
Rows: 153120
Columns: 178
Missing common columns: 0

Dataset 2022
Rows: 159173
Columns: 169
Missing common columns: 0

Audit connection check completed.


# SECTION 6: Create Working Copies

In [ ]:
# Raw datasets remain unchanged.
# All preprocessing operations are performed on working copies.
data_2018 = df_2018.copy()
data_2021 = df_2021.copy()
data_2022 = df_2022.copy()

print("Working copies created.")

Working copies created.


# SECTION 7: Define Target and Identifier

In [ ]:
# Target:abandono_hash
# Identifier:dni_hash
# The anonymized student identifier is retained for longitudinal
# grouping but will NOT be used as a predictive feature.

TARGET_COLUMN = "abandono_hash"

STUDENT_ID_COLUMN = "dni_hash"

print("Target column:", TARGET_COLUMN)
print("Student identifier:", STUDENT_ID_COLUMN)

for dataset_name, df in {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}.items():

    assert TARGET_COLUMN in df.columns
    assert STUDENT_ID_COLUMN in df.columns

print("\nTarget and identifier verified.")

Target column: abandono_hash
Student identifier: dni_hash

Target and identifier verified.


# SECTION 8: Normalize Target Encoding

In [ ]:
# UPV target:
#   A = dropout / withdrawal
#   B = non-dropout
#
# We preserve the original target column and create a numeric
# target column for machine learning.
TARGET_MAPPING = {
    "A": 1,
    "B": 0
}

for dataset_name, df in {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}.items():

    unique_targets = set(
        df[TARGET_COLUMN].dropna().unique()
    )

    print(dataset_name, "target values:", unique_targets)

    unexpected = unique_targets - set(TARGET_MAPPING.keys())

    if unexpected:
        raise ValueError(
            f"Unexpected target values in {dataset_name}: {unexpected}"
        )

    df["target"] = df[TARGET_COLUMN].map(TARGET_MAPPING)

print("\nNumeric target created successfully.")

print("\n2018:")
print(data_2018["target"].value_counts())

print("\n2021:")
print(data_2021["target"].value_counts())

print("\n2022:")
print(data_2022["target"].value_counts())

# SECTION 9: Identify Leakage-Sensitive Variables

In [ ]:
# Features that may contain information about the outcome itself
# must not be used as predictors.

LEAKAGE_COLUMNS = [
    "baja_fecha"
]

# Dataset-level timestamp
NON_PREDICTIVE_METADATA_COLUMNS = [
    "fecha_datos"
]

print("Leakage-sensitive columns:")
for col in LEAKAGE_COLUMNS:
    print("-", col)

print("\nNon-predictive metadata:")
for col in NON_PREDICTIVE_METADATA_COLUMNS:
    print("-", col)

Leakage-sensitive columns:
- baja_fecha

Non-predictive metadata:
- fecha_datos


# SECTION 10: Remove Identifier / Leakage from Predictive Features

In [ ]:
# These columns are retained in the working data where necessary
# for longitudinal processing, but they are excluded from the
# predictive feature matrix.
NON_PREDICTIVE_COLUMNS = [
    STUDENT_ID_COLUMN,
    TARGET_COLUMN,
    "target",
    *LEAKAGE_COLUMNS,
    *NON_PREDICTIVE_METADATA_COLUMNS
]

NON_PREDICTIVE_COLUMNS = list(
    dict.fromkeys(NON_PREDICTIVE_COLUMNS)
)

print("Non-predictive columns:")

for col in NON_PREDICTIVE_COLUMNS:
    print("-", col)

Non-predictive columns:
- dni_hash
- abandono_hash
- target
- baja_fecha
- fecha_datos


# SECTION 11: Separate Temporal / Digital Behaviour Features

In [ ]:
# The audit identified temporal/digital behaviour features using
# the feature-group classification.
temporal_features = (
    feature_summary[
        feature_summary["Feature Group"]
        == "Temporal / Digital Behavior"
    ]["Column"]
    .drop_duplicates()
    .tolist()
)

print("Total temporal/digital behaviour features:",
      len(temporal_features))

print("\nFirst 20 temporal features:")

for col in temporal_features[:20]:
    print("-", col)

Total temporal/digital behaviour features: 315

First 20 temporal features:
- pft_events_2018_9
- pft_days_logged_2018_9
- pft_visits_2018_9
- pft_assignment_submissions_2018_9
- pft_test_submissions_2018_9
- pft_total_minutes_2018_9
- n_wifi_days_2018_9
- resource_events_2018_9
- n_resource_days_2018_9
- pft_events_2018_10
- pft_days_logged_2018_10
- pft_visits_2018_10
- pft_assignment_submissions_2018_10
- pft_test_submissions_2018_10
- pft_total_minutes_2018_10
- n_wifi_days_2018_10
- resource_events_2018_10
- n_resource_days_2018_10
- pft_events_2018_11
- pft_days_logged_2018_11


# SECTION 12: Identify Common Predictive Features

In [ ]:
# Common columns are shared by all three datasets.
# This creates the initial candidate feature set.
#
# Important:
# This is NOT the final feature set.
# Feature selection and temporal engineering will happen later.
candidate_common_features = [
    col for col in common_columns
    if col not in NON_PREDICTIVE_COLUMNS
]

print("Common columns:", len(common_columns))
print(
    "Initial candidate predictive features:",
    len(candidate_common_features)
)

print("\nCandidate features:")

for col in candidate_common_features:
    print("-", col)

Common columns: 70
Initial candidate predictive features: 66

Candidate features:
- activ1
- actividades
- ajuste
- ajuste1
- anyo_ingreso
- anyo_inicio_estudios
- asi_hash
- asig1
- caca
- campus_hash
- cred_mat1
- cred_mat2
- cred_mat3
- cred_mat4
- cred_mat5
- cred_mat6
- cred_mat_anu
- cred_mat_movilidad
- cred_mat_normal
- cred_mat_practicas
- cred_mat_sem_a
- cred_mat_sem_b
- cred_mat_total
- cred_pend_sup_tit
- cred_ptes_acta
- cred_sup
- cred_sup_1o
- cred_sup_2o
- cred_sup_3o
- cred_sup_4o
- cred_sup_5o
- cred_sup_6o
- cred_sup_anu
- cred_sup_espec
- cred_sup_normal
- cred_sup_sem_a
- cred_sup_sem_b
- cred_sup_tit
- cred_sup_total
- curso_mas_alto
- curso_mas_bajo
- dedicacion
- desplazado_hash
- es_adaptado
- es_retitulado
- estudios_m_hash
- estudios_p_hash
- exento_npp
- grupos_por_tipocredito_hash
- impagado_curso_mat
- matricula_activa
- nota10_hash
- nota14_hash
- nota_asig_hash
- pract1
- practicas
- preferencia_seleccion
- rend_total_antepenultimo
- rend_total_penultim

# SECTION 13: Dataset-Specific Feature Analysis

In [ ]:
# Dataset-specific features are not automatically removed.
# Their relevance will be evaluated based on temporal structure
# and research objectives.
dataset_specific_summary = (
    dataset_specific_columns_df
    .groupby("Dataset")
    .size()
    .reset_index(name="Number_of_Dataset_Specific_Features")
)

display(dataset_specific_summary)

print("\nDataset-specific features will NOT be automatically discarded.")
print("They require temporal/research relevance evaluation.")

,Dataset,Number_of_Dataset_Specific_Features
0,2018,108
1,2021,108
2,2022,99



Dataset-specific features will NOT be automatically discarded.
They require temporal/research relevance evaluation.


# SECTION 14: Remove Exact Duplicate Rows

In [ ]:
# Exact duplicate rows are checked separately for each dataset.
# They are removed only if they are truly identical across all columns.
duplicate_summary = []

for dataset_name, df in {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}.items():

    duplicate_count = int(
        df.duplicated().sum()
    )

    duplicate_summary.append({
        "Dataset": dataset_name,
        "Duplicate_Rows": duplicate_count
    })

    print(
        f"{dataset_name}: "
        f"{duplicate_count} exact duplicate rows"
    )

duplicate_summary_df = pd.DataFrame(
    duplicate_summary
)

display(duplicate_summary_df)

2018: 0 exact duplicate rows
2021: 0 exact duplicate rows
2022: 0 exact duplicate rows


,Dataset,Duplicate_Rows
0,2018,0
1,2021,0
2,2022,0


# SECTION 15: Data Type Normalization

In [ ]:
# Convert date columns to datetime format.
# Numerical columns are kept numerical.
DATE_COLUMNS = [
    "baja_fecha",
    "fecha_datos"
]

for df in [data_2018, data_2021, data_2022]:

    for col in DATE_COLUMNS:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

print("Date columns converted to datetime.")

Date columns converted to datetime.


# SECTION 16: Missing Value Profile for Candidate Features

In [ ]:
# Missing values are profiled before deciding how each feature should be treated.
missing_profile = []

for dataset_name, df in {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}.items():

    for column in candidate_common_features:

        if column not in df.columns:
            continue

        missing_count = int(
            df[column].isna().sum()
        )

        missing_percent = (
            df[column].isna().mean() * 100
        )

        missing_profile.append({
            "Dataset": dataset_name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percent": round(
                missing_percent,
                2
            )
        })


missing_profile_df = pd.DataFrame(
    missing_profile
)

display(
    missing_profile_df
    .sort_values(
        "Missing_Percent",
        ascending=False
    )
)

,Dataset,Column,Missing_Count,Missing_Percent
61,2018,rendimiento_cuat_b,152446,100.00
60,2018,rendimiento_cuat_a,152446,100.00
47,2018,exento_npp,152446,100.00
62,2018,rendimiento_total,152446,100.00
176,2022,es_retitulado,158630,99.66
...,...,...,...,...
169,2022,cred_sup_tit,0,0.00
170,2022,cred_sup_total,0,0.00
182,2022,matricula_activa,0,0.00
187,2022,practicas,0,0.00


# SECTION 17: Define Missingness Categories

In [ ]:
# These categories are used for investigation.
# They are NOT automatic deletion rules.
def classify_missingness(percent):

    if percent == 0:
        return "No Missing"

    elif percent < 20:
        return "Low Missingness"

    elif percent < 50:
        return "Moderate Missingness"

    elif percent < 80:
        return "High Missingness"

    else:
        return "Very High Missingness"


missing_profile_df["Missingness_Category"] = (
    missing_profile_df["Missing_Percent"]
    .apply(classify_missingness)
)

display(
    missing_profile_df.head(20)
)

,Dataset,Column,Missing_Count,Missing_Percent,Missingness_Category
0,2018,activ1,147848,96.98,Very High Missingness
1,2018,actividades,134896,88.49,Very High Missingness
2,2018,ajuste,146950,96.39,Very High Missingness
3,2018,ajuste1,0,0.00,No Missing
4,2018,anyo_ingreso,32699,21.45,Moderate Missingness
5,2018,anyo_inicio_estudios,0,0.00,No Missing
6,2018,asi_hash,0,0.00,No Missing
7,2018,asig1,0,0.00,No Missing
8,2018,caca,0,0.00,No Missing
9,2018,campus_hash,0,0.00,No Missing


# SECTION 18: Identify High-Missingness Features

In [ ]:
# Features with very high missingness are flagged for later feature-selection decisions.
# They are NOT deleted automatically.
HIGH_MISSING_THRESHOLD = 80.0

high_missing_features = (
    missing_profile_df[
        missing_profile_df["Missing_Percent"]
        >= HIGH_MISSING_THRESHOLD
    ]
)

print(
    "Feature-dataset combinations with "
    f">= {HIGH_MISSING_THRESHOLD}% missingness:"
)

display(high_missing_features)

Feature-dataset combinations with >= 80.0% missingness:


,Dataset,Column,Missing_Count,Missing_Percent,Missingness_Category
0,2018,activ1,147848,96.98,Very High Missingness
1,2018,actividades,134896,88.49,Very High Missingness
2,2018,ajuste,146950,96.39,Very High Missingness
43,2018,es_adaptado,147743,96.91,Very High Missingness
44,2018,es_retitulado,151439,99.34,Very High Missingness
47,2018,exento_npp,152446,100.00,Very High Missingness
49,2018,impagado_curso_mat,149188,97.86,Very High Missingness
54,2018,pract1,137362,90.11,Very High Missingness
60,2018,rendimiento_cuat_a,152446,100.00,Very High Missingness
61,2018,rendimiento_cuat_b,152446,100.00,Very High Missingness


# SECTION 19: Prepare Longitudinal Data

In [ ]:
# UPV contains multiple records per student.
# Therefore, student-level identity is preserved for longitudinal processing.
for dataset_name, df in {
    "2018": data_2018,
    "2021": data_2021,
    "2022": data_2022
}.items():

    student_counts = (
        df.groupby(STUDENT_ID_COLUMN)
        .size()
        .reset_index(name="Record_Count")
    )

    print(
        f"{dataset_name}: "
        f"{student_counts.shape[0]} unique students"
    )

    print(
        "Mean records/student:",
        round(
            student_counts["Record_Count"].mean(),
            2
        )
    )

2018: 20233 unique students
Mean records/student: 7.53
2021: 20264 unique students
Mean records/student: 7.56
2022: 20427 unique students
Mean records/student: 7.79


# SECTION 20: TEMPORAL FEATURE INVENTORY

In [ ]:
# Extract temporal feature names and identify their year/month components.
import re


def parse_temporal_feature(column):

    pattern = (
        r"^(pft_events|pft_visits|pft_days_logged|"
        r"pft_assignment_submissions|pft_test_submissions|"
        r"pft_total_minutes|resource_events|"
        r"n_resource_days|n_wifi_days)_(\d{4})_(\d{1,2})$"
    )

    match = re.match(pattern, column)

    if match:

        return {
            "Column": column,
            "Feature_Type": match.group(1),
            "Year": int(match.group(2)),
            "Month": int(match.group(3))
        }

    return None


temporal_inventory_rows = []

for column in temporal_features:

    parsed = parse_temporal_feature(column)

    if parsed is not None:
        temporal_inventory_rows.append(parsed)


temporal_inventory = pd.DataFrame(
    temporal_inventory_rows
)

display(temporal_inventory.head())

print(
    "\nTemporal features parsed:",
    len(temporal_inventory)
)

,Column,Feature_Type,Year,Month
0,pft_events_2018_9,pft_events,2018,9
1,pft_days_logged_2018_9,pft_days_logged,2018,9
2,pft_visits_2018_9,pft_visits,2018,9
3,pft_assignment_submissions_2018_9,pft_assignment_submissions,2018,9
4,pft_test_submissions_2018_9,pft_test_submissions,2018,9



Temporal features parsed: 315


# SECTION 21: Temporal Feature Coverage

In [ ]:
# Show which months and feature types are available in each dataset.
if not temporal_inventory.empty:

    temporal_coverage = (
        temporal_inventory
        .groupby(
            ["Year", "Month"]
        )
        .size()
        .reset_index(
            name="Number_of_Features"
        )
        .sort_values(
            ["Year", "Month"]
        )
    )

    display(temporal_coverage)

,Year,Month,Number_of_Features
0,2018,9,9
1,2018,10,9
2,2018,11,9
3,2018,12,9
4,2019,1,9
5,2019,2,9
6,2019,3,9
7,2019,4,9
8,2019,5,9
9,2019,6,9


# SECTION 22: Create Initial Preprocessing Metadata

In [ ]:
# Record the preprocessing decisions made before actual transformation.
preprocessing_decisions = pd.DataFrame([
    {
        "Item": "Target",
        "Column": TARGET_COLUMN,
        "Action": "Encode A=1, B=0",
        "Reason": "Binary dropout prediction target"
    },
    {
        "Item": "Student Identifier",
        "Column": STUDENT_ID_COLUMN,
        "Action": "Retain for longitudinal grouping; exclude from model",
        "Reason": "Anonymized identifier"
    },
    {
        "Item": "Leakage",
        "Column": "baja_fecha",
        "Action": "Exclude from predictive features",
        "Reason": "May directly reveal withdrawal outcome"
    },
    {
        "Item": "Metadata",
        "Column": "fecha_datos",
        "Action": "Exclude from predictive features",
        "Reason": "Dataset-level timestamp"
    },
    {
        "Item": "Temporal Features",
        "Column": "Temporal / Digital Behavior",
        "Action": "Preserve for temporal feature engineering",
        "Reason": "Longitudinal behavioural information"
    },
    {
        "Item": "High Missingness",
        "Column": "Features >= 80% missing",
        "Action": "Flag for review",
        "Reason": "Do not automatically delete before feature analysis"
    },
    {
        "Item": "Dataset-Specific Features",
        "Column": "Dataset-specific columns",
        "Action": "Retain temporarily",
        "Reason": "Require temporal comparability assessment"
    }
])

display(preprocessing_decisions)

,Item,Column,Action,Reason
0,Target,abandono_hash,"Encode A=1, B=0",Binary dropout prediction target
1,Student Identifier,dni_hash,Retain for longitudinal grouping; exclude from...,Anonymized identifier
2,Leakage,baja_fecha,Exclude from predictive features,May directly reveal withdrawal outcome
3,Metadata,fecha_datos,Exclude from predictive features,Dataset-level timestamp
4,Temporal Features,Temporal / Digital Behavior,Preserve for temporal feature engineering,Longitudinal behavioural information
5,High Missingness,Features >= 80% missing,Flag for review,Do not automatically delete before feature ana...
6,Dataset-Specific Features,Dataset-specific columns,Retain temporarily,Require temporal comparability assessment


# SECTION 23: Save Preprocessing Metadata

In [ ]:
preprocessing_decisions.to_csv(
    PREPROCESSING_RESULT_DIR / "preprocessing_decisions.csv",
    index=False
)

missing_profile_df.to_csv(
    PREPROCESSING_RESULT_DIR / "missing_profile.csv",
    index=False
)

temporal_inventory.to_csv(
    PREPROCESSING_RESULT_DIR / "temporal_inventory.csv",
    index=False
)

print("Preprocessing metadata saved successfully.")

print("\nSaved files:")

for file in sorted(
    PREPROCESSING_RESULT_DIR.glob("*.csv")
):
    print("-", file.name)

Preprocessing metadata saved successfully.

Saved files:
- missing_profile.csv
- preprocessing_decisions.csv
- temporal_inventory.csv


# SECTION 24: Save Intermediate Clean Working Data

In [ ]:
# These files are intermediate preprocessing outputs.
# Raw datasets remain untouched in Datasets/UPV/.
#
# Note:
# Temporal aggregation and final feature engineering are NOT
# performed yet.
INTERMEDIATE_DIR = PREPROCESSING_RESULT_DIR / "Intermediate"

INTERMEDIATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save working copies
data_2018.to_pickle(
    INTERMEDIATE_DIR / "upv_2018_preprocessing_stage.pkl"
)

data_2021.to_pickle(
    INTERMEDIATE_DIR / "upv_2021_preprocessing_stage.pkl"
)

data_2022.to_pickle(
    INTERMEDIATE_DIR / "upv_2022_preprocessing_stage.pkl"
)

print("Intermediate working datasets saved.")
print("Location:", INTERMEDIATE_DIR)

Intermediate working datasets saved.
Location: /content/drive/MyDrive/Thesis Fall 2025 (DG1)/Code Implementation/Final/Results/UPV/Preprocessing/Intermediate


# SECTION 25: Final Preprocessing Status

In [ ]:
print("=" * 70)
print("UPV PREPROCESSING STAGE STATUS")
print("=" * 70)

print("""
Completed:

1. Raw UPV datasets loaded
2. Audit results connected
3. Target validated and encoded
4. Student identifier preserved for longitudinal processing
5. Leakage-sensitive variables identified
6. Dataset-level metadata identified
7. Common candidate features identified
8. Dataset-specific features retained for evaluation
9. Missingness profiled
10. Temporal features identified
11. Temporal feature inventory created
12. Preprocessing decisions documented
13. Intermediate working datasets saved

Not yet performed:

1. Final temporal aggregation
2. Final feature selection
3. Missing-value imputation
4. Categorical encoding
5. Numerical scaling
6. Train/validation/test split
7. Model training

These steps should be performed only after temporal feature
engineering and leakage-safe feature construction.
""")

print("=" * 70)
print("02_UPV_Preprocessing.ipynb INITIAL STAGE COMPLETE")
print("=" * 70)

UPV PREPROCESSING STAGE STATUS

Completed:

1. Raw UPV datasets loaded
2. Audit results connected
3. Target validated and encoded
4. Student identifier preserved for longitudinal processing
5. Leakage-sensitive variables identified
6. Dataset-level metadata identified
7. Common candidate features identified
8. Dataset-specific features retained for evaluation
9. Missingness profiled
10. Temporal features identified
11. Temporal feature inventory created
12. Preprocessing decisions documented
13. Intermediate working datasets saved

Not yet performed:

1. Final temporal aggregation
2. Final feature selection
3. Missing-value imputation
4. Categorical encoding
5. Numerical scaling
6. Train/validation/test split
7. Model training

These steps should be performed only after temporal feature
engineering and leakage-safe feature construction.

02_UPV_Preprocessing.ipynb INITIAL STAGE COMPLETE
